# Python API

This tutorial runs scPertEval end-to-end from Python — no CLI, no output files required.
It mirrors the [CLI walkthrough](01_cli_walkthrough.ipynb) on the same tiny synthetic
dataset, using the native [Python API](../user-guide/python-api.md). It is a runnable
notebook (use the download/Colab buttons above).

The API is always **prepare, then run**: build a reusable handle for a dataset with
`prepare`, then call `calibrate` / `score` / `de` on that handle — each returns its result
as in-memory pandas.

In [1]:
# In Colab or a fresh environment, install scPertEval first:
# %pip install scperteval

## Set up a dataset

We build a 60-gene toy dataset with a control population and four perturbations, each with
its own distinct block of up-regulated genes. The Python API accepts an in-memory
[AnnData](https://anndata.readthedocs.io/) directly, so — unlike the CLI — we never have to
write it to disk:

In [2]:
import anndata as ad
import numpy as np

rng = np.random.default_rng(0)
ng, n_ctrl, n_pert = 60, 150, 120
de_genes = {"pertA": range(0, 6), "pertB": range(15, 21), "pertC": range(30, 36), "pertD": range(45, 51)}

parts = [rng.poisson(1.0, (n_ctrl, ng)).astype(np.float32)]
labels = ["control"] * n_ctrl
for name, genes in de_genes.items():
    x = rng.poisson(1.0, (n_pert, ng)).astype(np.float32)
    x[:, list(genes)] += 6.0  # up-regulate this perturbation's marker genes
    parts.append(x)
    labels += [name] * n_pert

adata = ad.AnnData(np.vstack(parts))
adata.var_names = [f"g{i}" for i in range(ng)]
adata.obs["perturbation"] = labels
adata

AnnData object with n_obs × n_vars = 630 × 60
    obs: 'perturbation'
    layers: None (.X)

## Prepare the dataset

{func}`scperteval.prepare <scperteval.api.prepare>` reads and indexes the dataset once and
precomputes the feature spaces the protocols we name will need. It returns a reusable
{class}`~scperteval.api.Prepared` handle that we pass to every verb below — they share its
dataset and caches (no reload), and are even safe to call concurrently. We keep the run
small and deterministic here (`subsample`, `seed`, `min_cells`, `workers`):

In [3]:
import scperteval as sp

prep = sp.prepare(adata, ["pearson_ctrl", "mse", "de_auprc"], subsample=400, seed=0, min_cells=10, workers=1)
prep

Prepared(name='dataset', perturbations=4)

## Calibrate a protocol against built-in controls

{func}`scperteval.calibrate <scperteval.api.calibrate>` scores **one** protocol against a
positive and a negative control and returns a calibrated **DRF** (or **BDS**). It returns an
{class}`~scperteval.api.EvalResult`, whose `.aggregate` holds the protocol's summary stats:

In [4]:
res = sp.calibrate(prep, "pearson_ctrl")
res.aggregate

{'mean': 0.9247784193453554, 'median': 0.9236923538884874}

`.per_perturbation` holds the underlying per-perturbation table — the raw control values
and the calibrated DRF — identical to the CSV the CLI writes:

In [5]:
res.per_perturbation

,protocol,perturbation,raw_positive,raw_negative,drf,dataset,de_method,subsample,seed
0,pearson_ctrl,pertA,0.905564,-0.231327,0.923305,dataset,t-test,400,0
1,pearson_ctrl,pertB,0.897727,-0.210698,0.915525,dataset,t-test,400,0
2,pearson_ctrl,pertC,0.907471,-0.218775,0.924080,dataset,t-test,400,0
3,pearson_ctrl,pertD,0.922935,-0.208009,0.936204,dataset,t-test,400,0


The same handle is reusable: calibrate another protocol on it, this time asking for the
**BDS** calibrator with `calibrator="bds"`:

In [6]:
sp.calibrate(prep, "mse", calibrator="bds").aggregate

{'bds': 1.0}

## Score predictions against ground truth

{func}`scperteval.score <scperteval.api.score>` compares a model's predicted cells to the
real ones for **one** protocol; the argument order is `(prepared, protocol, predictions)`.
We fabricate a "degraded" prediction (the perturbed cells shrunk toward the control mean
plus noise) to stand in for a model's output:

In [7]:
pert = np.asarray(adata.obs["perturbation"]).astype(str)
pred = adata[pert != "control"].copy()
ctrl_mean = np.asarray(adata.X[pert == "control"]).mean(0)
pred.X = np.clip(0.4 * pred.X + 0.6 * ctrl_mean + rng.normal(0, 0.2, pred.X.shape), 0, None).astype(np.float32)

res = sp.score(prep, "pearson", pred)
res.aggregate

{'mean': 0.997633093513851, 'median': 0.9975959519069415}

In [8]:
res.per_perturbation

,protocol,perturbation,raw_prediction,score,dataset,de_method,subsample,seed
0,pearson,pertA,0.997524,0.997524,dataset,t-test,400,0
1,pearson,pertB,0.997668,0.997668,dataset,t-test,400,0
2,pearson,pertC,0.997882,0.997882,dataset,t-test,400,0
3,pearson,pertD,0.997458,0.997458,dataset,t-test,400,0


## Differential expression

{func}`scperteval.de <scperteval.api.de>` returns the per-gene ground-truth DE for **one**
method as a {class}`~scperteval.api.DatasetDEResults` — a `NamedTuple` of two
perturbations × genes DataFrames (`.statistic` and `.pvalue_adj`):

In [9]:
d = sp.de(prep, "t-test")
d.statistic.iloc[:, :6]  # t-statistic, first six genes

/Users/zbo/Documents/opensource.nosync/scPertEval/.claude/worktrees/agent-a84d558a06d863078/src/scperteval/context.py:259: UserWarning: Excluding 'pertA' removes 99/400 (25%) of the reference sample -- this perturbation is a large share of the dataset, so its leave-one-out reference is much smaller than other perturbations'. Raise --subsample (or expect noisier single-cell metrics for this perturbation).
  keep = self.reference().keep(pert)
/Users/zbo/Documents/opensource.nosync/scPertEval/.claude/worktrees/agent-a84d558a06d863078/src/scperteval/context.py:259: UserWarning: Excluding 'pertB' removes 100/400 (25%) of the reference sample -- this perturbation is a large share of the dataset, so its leave-one-out reference is much smaller than other perturbations'. Raise --subsample (or expect noisier single-cell metrics for this perturbation).
  keep = self.reference().keep(pert)
/Users/zbo/Documents/opensource.nosync/scPertEval/.claude/worktrees/agent-a84d558a06d863078/src/scperteval/co

,g0,g1,g2,g3,g4,g5
pertA,42.066928,44.686031,43.902499,41.186745,42.446310,38.946920
pertB,-9.174290,-9.554969,-9.174856,-10.411350,-8.694524,-9.776829
pertC,-8.410386,-8.231241,-9.799363,-8.861210,-8.967629,-9.531205
pertD,-10.093081,-9.972281,-7.766397,-7.825244,-9.817560,-9.445006


The blocks of up-regulated genes we built into each perturbation show up as the large
positive statistics above. The result unpacks like any `NamedTuple`, and a different method
reuses the same prepared dataset (cached separately, no reload):

In [10]:
statistic, pvalue_adj = sp.de(prep, "MWU")
pvalue_adj.iloc[:, :6]

/Users/zbo/Documents/opensource.nosync/scPertEval/.claude/worktrees/agent-a84d558a06d863078/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-16 18:20:54.264 | INFO     | illico.asymptotic_wilcoxon:asymptotic_wilcoxon:412 - No precompilation needed for Rust kernels.


2026-07-16 18:20:54.265 | INFO     | illico.asymptotic_wilcoxon:asymptotic_wilcoxon:425 - Found 2 unique groups (min size: 60 cells; max size: 301 cells), with reference group: reference


  0%|          | 0.00/120 [00:00<?, ?it/s]

100%|██████████| 120/120 [00:00<00:00, 174kit/s]


2026-07-16 18:20:54.274 | INFO     | illico.asymptotic_wilcoxon:asymptotic_wilcoxon:412 - No precompilation needed for Rust kernels.


...


2026-07-16 18:20:54.280 | INFO     | illico.asymptotic_wilcoxon:asymptotic_wilcoxon:412 - No precompilation needed for Rust kernels.


2026-07-16 18:20:54.280 | INFO     | illico.asymptotic_wilcoxon:asymptotic_wilcoxon:425 - Found 2 unique groups (min size: 60 cells; max size: 299 cells), with reference group: reference


  0%|          | 0.00/120 [00:00<?, ?it/s]

100%|██████████| 120/120 [00:00<00:00, 420kit/s]

,g0,g1,g2,g3,g4,g5
pertA,2.797079e-35,2.797079e-35,2.797079e-35,2.797079e-35,2.797079e-35,2.797079e-35
pertB,6.376472e-04,1.197092e-05,1.767750e-04,1.701615e-05,1.767750e-04,4.620505e-05
pertC,1.170022e-03,1.311963e-02,2.439663e-05,3.419037e-05,1.674677e-04,5.807614e-04
pertD,5.041424e-06,2.143007e-05,2.758781e-03,8.494346e-03,7.579985e-05,5.890433e-05


## Where to next

- The [Python API guide](../user-guide/python-api.md) and [API reference](../api/api.md) for
  the full signatures and options (`out_dir=`, `calibrator="bds"`, tunable protocols, …).
- [Calibration](../user-guide/calibration.md) for what DRF and BDS actually measure.
- The [CLI walkthrough](01_cli_walkthrough.ipynb) for the equivalent command-line flow.